# Sistema de Recomendação - Proof of Concept

A ideia aqui é ja começarmos a fazer umas recomendações com base só em distribuição de renda fixa e renda variável para vermos se encontramos bons resultados e também servir como âncora de entrega caso o resultado final não fosse alcançado. Esse arquivo não é necessário pro andamento do projeto, guardamos por formalidade de manter um histórico de desenvolvimento.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
barema_por_ativo = pd.read_csv('../data/processed/barema_por_ativo.csv')
barema_por_ativo

,NOME ATIVO,CLASSIFICACAO DO ATIVO
0,1832421D CN,Renda Variável
1,1SANPN,Renda Variável
2,2299955D CN,Renda Variável
3,2458693D US,Renda Variável
4,700 HK,Renda Variável
...,...,...
8041,ZAVI11,Renda Variável
8042,ZIFI11,Renda Variável
8043,ZS US,Renda Variável
8044,ZTO US,Renda Variável


In [3]:
percentual_de_distribuicao = pd.read_csv('../data/processed/percentual_de_distribuicao.csv')
percentual_de_distribuicao

,Conta,Renda Fixa,Renda Variável
0,27,100.0,0.0
1,44,100.0,0.0
2,103,100.0,0.0
3,285,100.0,0.0
4,299,100.0,0.0
...,...,...,...
6096,6150189,0.0,100.0
6097,6167502,100.0,0.0
6098,6178808,100.0,0.0
6099,6184909,100.0,0.0


In [4]:
user_item_binario = pd.read_csv('../data/processed/user_item_binario.csv')
user_item_binario

,CONTA,1832421D CN,1SANPN,2299955D CN,2458693D US,700 HK,8001 JP,8031 JP,8058 JP,9988 HK,...,YUFI11,Z US.,Z1OM34,ZAGH11,ZAMP3,ZAVI11,ZIFI11,ZS US,ZTO US,ZTS US
0,27,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,44,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,103,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,285,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,299,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6097,6150189,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6098,6167502,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6099,6178808,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6100,6184909,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Aqui ainda estavamos em uma etapa de discovery e passamos apenas os ativos que o usuário tem, e não quanto esse usuário investe em cada ativo pra fazer a distribuição por ativo. Foi a partir daqui que percebemos que precisariamos fazer a distribuição de renda em cada tipo de investimento e posteriormente veio a ideia de granular em subgrupos!

In [ ]:
def recomendar_ativos(id_novo_usuario, ativos_usuario, dist_df, user_item_binario, barema_df, k_vizinhos=2):
    dist_novo = (
        barema_df[barema_df['NOME ATIVO'].isin(ativos_usuario)]
        .groupby('CLASSIFICACAO DO ATIVO')
        .size()
        .reindex(['Renda Fixa', 'Renda Variável'], fill_value=0)
        .astype(float)
    )
    dist_novo = (dist_novo / dist_novo.sum())*100
    dist_novo.name = id_novo_usuario

    dist_all = dist_df.set_index('Conta').copy()
    dist_all.loc[id_novo_usuario] = dist_novo


    similarity_matrix = cosine_similarity(dist_all.fillna(0))
    similarity_df = pd.DataFrame(similarity_matrix, index=dist_all.index, columns=dist_all.index)

    vizinhos = (
        similarity_df[id_novo_usuario]
        .drop(index=id_novo_usuario)
        .sort_values(ascending=False)
        .head(k_vizinhos)
        .index
    )

    
    user_bin = pd.Series(0, index=user_item_binario.columns[1:], name=id_novo_usuario) 
    user_bin.loc[[a for a in ativos_usuario if a in user_bin.index]] = 1


    user_item_ext = user_item_binario.copy()
    

    user_item_ext_indexed = user_item_ext.set_index('CONTA')
    
    vizinhos_validos = [v for v in vizinhos if v in user_item_ext_indexed.index]
    
    if not vizinhos_validos:
        print("Nenhum dos vizinhos similares existe na matriz user-item. Usando todos os usuários para recomendação.")

        vizinhos_matrix = user_item_ext_indexed.copy()
    else:
        print(f"Vizinhos válidos encontrados: {vizinhos_validos}")
        vizinhos_matrix = user_item_ext_indexed.loc[vizinhos_validos]
    score_ativos = vizinhos_matrix.mean(axis=0)

    score_ativos = score_ativos[user_bin == 0]

    recomendados = score_ativos.sort_values(ascending=False).head(5)

    return recomendados

In [6]:
id_novo = 999
ativos = ['1SANPN', '1832421D CN']  # ativos que esse usuário tem

recomendados = recomendar_ativos(id_novo, ativos, percentual_de_distribuicao, user_item_binario, barema_por_ativo)
print(recomendados)


Vizinhos válidos encontrados: [6150189, 6069730]
BRTYON    0.5
CSMG3     0.5
DIRR3     0.5
YDUQ3     0.5
BPAC11    0.5
dtype: float64
